# 05 — Upload to BigQuery
**MSBA 305 | Maritime Shipping Intelligence Pipeline**

Uploads all 4 tables to Google BigQuery, creates SQL views, and takes snapshots.

---
**Prerequisites:**
1. Create a Google Cloud project (console.cloud.google.com)
2. Enable BigQuery API
3. Create a dataset named `shipping_data`
4. Download a service account JSON key
5. Upload the JSON key to your Google Drive as `repo/gcp_key.json`

In [ ]:
import os, sys
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
from google.oauth2 import service_account

# ── Environment detection: works in Colab and local/GitHub clone ──────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/repo'
    print('Running in Google Colab')
except ImportError:
    BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f'Running locally — BASE: {BASE}')

!pip install google-cloud-bigquery pandas-gbq pyarrow db-dtypes --quiet

CLEAN_DIR  = os.path.join(BASE, 'data', 'clean')

# ── YOUR CONFIG ─────────────────────────────────────────────────────
# In Colab: paste your values here
# Locally:  set environment variables BQ_PROJECT, BQ_DATASET, GOOGLE_APPLICATION_CREDENTIALS
BQ_PROJECT = os.environ.get('BQ_PROJECT',  'msba305-shipping')
BQ_DATASET = os.environ.get('BQ_DATASET',  'shipping_data')
# GCP key: in Colab upload gcp_key.json to repo root (same level as notebooks/)
#          locally set GOOGLE_APPLICATION_CREDENTIALS env var
GCP_KEY    = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS',
                             os.path.join(BASE, 'gcp_key.json'))
# ─────────────────────────────────────────────────────────────────────

if not os.path.exists(GCP_KEY):
    raise FileNotFoundError(
        f'GCP key not found at {GCP_KEY}.\n'
        'In Colab: upload gcp_key.json to MyDrive/repo/\n'
        'Locally:  set GOOGLE_APPLICATION_CREDENTIALS env var'
    )

credentials = service_account.Credentials.from_service_account_file(GCP_KEY)
client = bigquery.Client(project=BQ_PROJECT, credentials=credentials)

print(f'BigQuery client ready — project: {BQ_PROJECT}')


ValueError: mount failed

## 1. Take Snapshots Before Uploading (backup strategy from rubric Section 4.7)

In [ ]:
def take_snapshot(table_name):
    src  = f'{BQ_PROJECT}.{BQ_DATASET}.{table_name}'
    snap = f'{BQ_PROJECT}.{BQ_DATASET}.{table_name}_snap_{datetime.now().strftime("%Y%m%d")}'
    q = f"""
        CREATE SNAPSHOT TABLE `{snap}`
        CLONE `{src}`
        OPTIONS (expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 14 DAY))
    """
    try:
        client.query(q).result()
        print(f'  Snapshot created → {snap.split(".")[-1]} (expires 14 days)')
    except Exception as e:
        print(f'  Snapshot skipped ({table_name} may not exist yet): {str(e)[:60]}')

for t in ['trade_flows','bdi_daily','port_weather','shipping_combined']:
    take_snapshot(t)

## 2. Upload All Tables

In [ ]:
def upload_table(csv_path, table_name):
    if not os.path.exists(csv_path):
        print(f'  SKIP: {csv_path} not found')
        return
    df = pd.read_csv(csv_path, low_memory=False)
    table_id = f'{BQ_PROJECT}.{BQ_DATASET}.{table_name}'
    job_cfg  = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        autodetect=True,
    )
    try:
        job = client.load_table_from_dataframe(df, table_id, job_config=job_cfg)
        job.result()
        tbl = client.get_table(table_id)
        print(f'  {table_name}: {len(df):,} rows → BigQuery (total: {tbl.num_rows:,})')
    except Exception as e:
        print(f'  ERROR {table_name}: {e}')

# 5 tables: 4 clean sources + 1 combined master
tables = [
    ('un_comtrade_clean.csv',   'trade_flows'),
    ('bdi_clean.csv',           'bdi_daily'),
    ('port_weather_clean.csv',  'port_weather'),
    ('aisstream_clean.csv',     'vessel_movements'),
    ('shipping_combined.csv',   'shipping_combined'),
]

print('Uploading tables...')
for csv_file, table_name in tables:
    print(f'\n{table_name}:')
    upload_table(f'{CLEAN_DIR}/{csv_file}', table_name)


## 3. Create SQL View — Trade Balance

In [ ]:
view_sql = f"""
CREATE OR REPLACE VIEW `{BQ_PROJECT}.{BQ_DATASET}.v_trade_balance` AS
SELECT
    reporter_country,
    reporter_iso,
    CAST(hs_code AS STRING)  AS hs_code,
    commodity,
    year,
    ROUND(SUM(CASE WHEN flow_direction='Export' THEN trade_value_usd ELSE 0 END)/1e9,2) AS exports_B,
    ROUND(SUM(CASE WHEN flow_direction='Import' THEN trade_value_usd ELSE 0 END)/1e9,2) AS imports_B,
    ROUND(
        (SUM(CASE WHEN flow_direction='Export' THEN trade_value_usd ELSE 0 END)
       - SUM(CASE WHEN flow_direction='Import' THEN trade_value_usd ELSE 0 END))
    /1e9, 2) AS balance_B,
    CASE
        WHEN SUM(CASE WHEN flow_direction='Export' THEN trade_value_usd ELSE 0 END)
           > SUM(CASE WHEN flow_direction='Import' THEN trade_value_usd ELSE 0 END)
        THEN 'Net Exporter'
        ELSE 'Net Importer'
    END AS trade_position
FROM `{BQ_PROJECT}.{BQ_DATASET}.trade_flows`
GROUP BY 1,2,3,4,5
"""
try:
    client.query(view_sql).result()
    print('View created → v_trade_balance')
except Exception as e:
    print(f'View creation failed: {e}')


## 4. Verify Upload — Run a Test Query

In [ ]:
test_q = f"""
SELECT 'trade_flows'       AS tbl, COUNT(*) AS rows FROM `{BQ_PROJECT}.{BQ_DATASET}.trade_flows`
UNION ALL SELECT 'bdi_daily',        COUNT(*) FROM `{BQ_PROJECT}.{BQ_DATASET}.bdi_daily`
UNION ALL SELECT 'port_weather',     COUNT(*) FROM `{BQ_PROJECT}.{BQ_DATASET}.port_weather`
UNION ALL SELECT 'vessel_movements', COUNT(*) FROM `{BQ_PROJECT}.{BQ_DATASET}.vessel_movements`
UNION ALL SELECT 'shipping_combined',COUNT(*) FROM `{BQ_PROJECT}.{BQ_DATASET}.shipping_combined`
"""
try:
    result = client.query(test_q).to_dataframe()
    print('Table row counts in BigQuery:')
    print(result.to_string(index=False))
except Exception as e:
    print(f'Verification query failed: {e}')

print('\nAll done.')
print(f'Open BigQuery console: https://console.cloud.google.com/bigquery?project={BQ_PROJECT}')
